# 1. Phân tích proto và service theo từng loại tấn công

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Tìm dataset gốc
DATA_PATH = None
SEARCH_ROOTS = [Path.cwd().resolve(), *Path.cwd().resolve().parents]

for folder in SEARCH_ROOTS:
    candidates = [
        folder / "data" / "raw" / "RT_IOT2022.csv",
        folder / "result" / "preprocessing" / "raw data" / "RT_IOT2022.csv",
        folder / "final_IoT" / "sample_project" / "result" / "preprocessing" / "raw data" / "RT_IOT2022.csv",
    ]
    for candidate in candidates:
        if candidate.is_file():
            DATA_PATH = candidate
            break
    if DATA_PATH is not None:
        break

if DATA_PATH is None:
    raise FileNotFoundError("Khong tim thay RT_IOT2022.csv trong cau truc project")

SAMPLE_ROOT = next(
    (
        candidate
        for folder in SEARCH_ROOTS
        for candidate in (folder, folder / "final_IoT" / "sample_project")
        if (candidate / "project").is_dir() and (candidate / "result").is_dir()
    ),
    None,
)
if SAMPLE_ROOT is None:
    raise FileNotFoundError("Khong tim thay thu muc goc sample_project")

RUN_DIR = SAMPLE_ROOT / "result" / "eda" / "EDA_Thanhdai" / "04_Feature_Target_Analysis"
FIG_DIR = RUN_DIR / "images"
TABLE_DIR = RUN_DIR / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

target = "Attack_type"

# Phân tích quan hệ giữa thuộc tính phân loại và nhãn
categorical_features = ["proto", "service"]

fig, axes = plt.subplots(1, 2, figsize=(19, 8))

for ax, feature in zip(axes, categorical_features):

    # Số lượng bản ghi theo từng nhãn
    counts = pd.crosstab(df[target], df[feature])

    # Tỷ lệ trong từng lớp Attack_type
    percentages = pd.crosstab(
        df[target],
        df[feature],
        normalize="index"
    ) * 100

    print(f"\n=== {feature.upper()} THEO ATTACK_TYPE ===")
    display(percentages.round(2))

    percentages.to_csv(
        TABLE_DIR / f"04_{feature}_by_target.csv",
        encoding="utf-8-sig"
    )

    sns.heatmap(
        percentages,
        annot=True,
        fmt=".0f",
        cmap="Blues",
        ax=ax
    )

    ax.set_title(f"{feature} theo Attack_type")
    ax.set_xlabel(feature)
    ax.set_ylabel("Attack_type")

plt.tight_layout()

plt.savefig(
    FIG_DIR / "04_categorical_target_heatmap.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

### Nhận xét: Mối quan hệ giữa đặc trưng phân loại và nhãn

**Kết quả phân tích:**

- Lớp `DOS_SYN_Hping` sử dụng giao thức TCP
  và có `service = "-"` ở toàn bộ 94.659 mẫu.
- Khoảng 99,66% mẫu `MQTT_Publish` có
  `service = "mqtt"`.
- Khoảng 97,94% mẫu `DDOS_Slowloris`
  có `service = "http"`.
- Lớp `NMAP_UDP_SCAN` chủ yếu sử dụng UDP.
- Một số nhãn có phân bố giao thức và
  dịch vụ đặc trưng rõ rệt.

**Kết luận:**

Cần tiếp tục đánh giá khả năng phụ thuộc
của mô hình vào `proto` và `service`.

# 2. Phân tích đặc trưng số theo từng nhãn

In [ ]:
features = [
    "flow_duration",
    "flow_pkts_per_sec",
    "bwd_pkts_tot"
]

# Thứ tự lớp theo số lượng mẫu
class_order = df[target].value_counts().index.tolist()

# Trung vị và phân vị 90 của từng đặc trưng theo lớp
medians = (
    df.groupby(target)[features]
    .median()
    .add_suffix("_Median")
)

p90 = (
    df.groupby(target)[features]
    .quantile(0.90)
    .add_suffix("_P90")
)

class_statistics = pd.concat(
    [medians, p90],
    axis=1
).loc[class_order]

print("=== THỐNG KÊ ĐẶC TRƯNG THEO NHÃN ===")
display(class_statistics.round(4))

class_statistics.to_csv(
    TABLE_DIR / "04_numeric_features_by_class.csv",
    encoding="utf-8-sig"
)

# Lấy tối đa 400 mẫu mỗi lớp để vẽ.
# Các lớp dưới 400 mẫu được giữ toàn bộ.
rng = np.random.default_rng(42)

sample_indices = np.concatenate([
    rng.choice(
        group.index.to_numpy(),
        size=min(len(group), 400),
        replace=False
    )
    for _, group in df.groupby(target)
])

plot_data = df.loc[
    sample_indices,
    [target] + features
].copy()

# Chuyển thang chỉ để vẽ biểu đồ
for col in features:
    if (plot_data[col] < 0).any():
        raise ValueError(f"{col} chứa giá trị âm")

    plot_data[f"log1p_{col}"] = np.log1p(
        plot_data[col]
    )

fig, axes = plt.subplots(3, 1, figsize=(14, 17))

for ax, col in zip(axes, features):

    sns.boxplot(
        data=plot_data,
        x=f"log1p_{col}",
        y=target,
        order=class_order,
        ax=ax,
        fliersize=1
    )

    ax.set_title(f"Phân bố {col} theo Attack_type")
    ax.set_xlabel(f"log1p({col})")
    ax.set_ylabel("Attack_type")

plt.tight_layout()

plt.savefig(
    FIG_DIR / "04_numeric_features_by_class.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

### Nhận xét: Đặc trưng số theo từng lớp

**Kết quả phân tích:**

- `flow_duration` có sự khác biệt lớn giữa
  các loại tấn công.
- `MQTT_Publish` có trung vị `flow_duration`
  khoảng 44,07 giây.
- `DDOS_Slowloris` có trung vị khoảng
  20,09 giây.
- Trong khi đó, `DOS_SYN_Hping` có thời gian
  rất nhỏ, gần bằng 0.

- Đối với `flow_pkts_per_sec`, lớp
  `DOS_SYN_Hping` có trung vị khoảng
  493.448 gói/giây.
- `MQTT_Publish` chỉ có trung vị khoảng
  0,35 gói/giây.

**Kết luận:**

Một số đặc trưng số có khả năng cung cấp
thông tin hữu ích cho việc phân loại.

Tuy nhiên, chưa thể kết luận đặc trưng
nào là tốt nhất chỉ dựa vào Boxplot.

# 3. Kiểm tra ngoại lệ theo từng loại tấn công

In [ ]:
feature = "flow_duration"

# Tính IQR trên toàn dataset để khảo sát
q1 = df[feature].quantile(0.25)
q3 = df[feature].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

# Đánh dấu mẫu nằm ngoài ngưỡng IQR toàn cục
outlier_mask = (
    (df[feature] < lower) |
    (df[feature] > upper)
)

# Tổng hợp tỷ lệ ngoại lệ theo lớp
outliers_by_class = (
    df.assign(is_outlier=outlier_mask)
    .groupby(target)["is_outlier"]
    .agg(["sum", "count", "mean"])
)

outliers_by_class.columns = [
    "Outlier Count",
    "Total Samples",
    "Outlier Ratio"
]

outliers_by_class["Outlier (%)"] = (
    outliers_by_class["Outlier Ratio"] * 100
)

outliers_by_class = outliers_by_class.sort_values(
    "Outlier (%)",
    ascending=False
)

print("=== NGƯỠNG IQR ===")
print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower:", lower)
print("Upper:", upper)

print("\n=== NGOẠI LỆ THEO TỪNG NHÃN ===")
display(outliers_by_class.round(2))

plt.figure(figsize=(12, 6))

sns.barplot(
    data=outliers_by_class.reset_index(),
    x="Outlier (%)",
    y=target,
    color="steelblue"
)

plt.title(
    "Tỷ lệ ngoại lệ flow_duration theo Attack_type"
)
plt.xlabel("Tỷ lệ mẫu nằm ngoài ngưỡng IQR (%)")
plt.ylabel("Loại tấn công")
plt.xlim(0, 100)

plt.tight_layout()

plt.savefig(
    FIG_DIR / "04_outliers_by_class.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

outliers_by_class.to_csv(
    TABLE_DIR / "04_outliers_by_class.csv",
    encoding="utf-8-sig"
)

### Nhận xét: Ngoại lệ theo từng loại tấn công

**Kết quả phân tích:**

Khi sử dụng ngưỡng IQR toàn cục
cho đặc trưng `flow_duration`:

- Khoảng 99,8% mẫu `MQTT_Publish`
  nằm ngoài ngưỡng IQR.
- Khoảng 99,4% mẫu `DDOS_Slowloris`
  nằm ngoài ngưỡng IQR.
- Khoảng 97,8% mẫu `Thing_Speak`
  nằm ngoài ngưỡng IQR.
- Trong khi đó, tỷ lệ tương ứng của
  `DOS_SYN_Hping` chỉ khoảng 0,1%.

**Kết luận:**

Không nên tự động loại bỏ ngoại lệ bằng
IQR trên toàn bộ dataset.

Đối với bài toán phân loại tấn công mạng,
các giá trị bất thường có thể chính là
thông tin quan trọng giúp nhận diện nhãn.


# 4. Kiểm tra sự phụ thuộc giữa cổng mạng và nhãn

In [ ]:
port_features = [
    "id.orig_p",
    "id.resp_p"
]

records = []

for feature in port_features:

    for label, group in df.groupby(target):

        counts = group[feature].value_counts()

        most_common_port = counts.index[0]
        most_common_count = counts.iloc[0]

        records.append({
            "Feature": feature,
            "Attack_type": label,
            "Most Common Port": most_common_port,
            "Count": most_common_count,
            "Total": len(group),
            "Share (%)": (
                most_common_count / len(group) * 100
            )
        })

port_summary = pd.DataFrame(records)

print("=== CỔNG NGUỒN ===")
display(
    port_summary[
        port_summary["Feature"] == "id.orig_p"
    ].sort_values(
        "Share (%)",
        ascending=False
    ).round(2)
)

print("\n=== CỔNG ĐÍCH ===")
display(
    port_summary[
        port_summary["Feature"] == "id.resp_p"
    ].sort_values(
        "Share (%)",
        ascending=False
    ).round(2)
)

destination_ports = port_summary[
    port_summary["Feature"] == "id.resp_p"
].sort_values("Share (%)", ascending=False)

plt.figure(figsize=(12, 6))

sns.barplot(
    data=destination_ports,
    x="Share (%)",
    y="Attack_type",
    color="steelblue"
)

plt.title(
    "Tỷ lệ sử dụng cổng đích phổ biến nhất theo nhãn"
)
plt.xlabel("Tỷ lệ trong từng lớp (%)")
plt.ylabel("Attack_type")
plt.xlim(0, 100)

plt.tight_layout()

plt.savefig(
    FIG_DIR / "04_destination_port_by_class.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

port_summary.to_csv(
    TABLE_DIR / "04_port_target_relationship.csv",
    index=False,
    encoding="utf-8-sig"
)

### Nhận xét: Mối quan hệ giữa cổng mạng và nhãn

**Kết quả phân tích:**

- Toàn bộ 94.659 mẫu `DOS_SYN_Hping`
  có cổng đích `id.resp_p = 21`.
- Khoảng 99,90% mẫu `MQTT_Publish`
  có cổng đích 1883.
- Khoảng 99,06% mẫu `DDOS_Slowloris`
  có cổng đích 80.
- Một số lớp khác có phân bố cổng
  đích đa dạng hơn.
  
**Kết luận:**

Cần nghiên cứu ảnh hưởng của `id.orig_p`
và `id.resp_p` đến chất lượng dự đoán.

Sự khác biệt kết quả giữa các trường hợp
có thể giúp đánh giá mức độ phụ thuộc
của mô hình vào cổng mạng.

# 5. Đánh giá mức độ liên hệ giữa đặc trưng và nhãn

In [ ]:
from sklearn.feature_selection import mutual_info_classif

rng = np.random.default_rng(42)

# Lấy mẫu theo từng lớp để giảm ảnh hưởng
# của lớp áp đảo và giảm thời gian tính toán
sample_indices = np.concatenate([
    rng.choice(
        group.index.to_numpy(),
        size=min(len(group), 800),
        replace=False
    )
    for _, group in df.groupby(target)
])

eda_sample = df.loc[sample_indices].copy()

# Chỉ khảo sát đặc trưng số
X_mi = eda_sample.select_dtypes(
    include=np.number
).drop(
    columns=["Unnamed: 0"],
    errors="ignore"
)

# Không đưa cột hằng vào tính MI
X_mi = X_mi.loc[
    :, X_mi.nunique(dropna=False) > 1
]

y_mi = eda_sample[target]

# Đánh dấu các đặc trưng số nguyên
# có rất ít giá trị khác nhau là rời rạc
discrete_mask = [
    pd.api.types.is_integer_dtype(X_mi[col])
    and X_mi[col].nunique() <= 20
    for col in X_mi.columns
]

mi_scores = mutual_info_classif(
    X_mi,
    y_mi,
    discrete_features=discrete_mask,
    random_state=42,
    n_neighbors=3
)

mi_result = pd.DataFrame({
    "Feature": X_mi.columns,
    "MI Score": mi_scores
}).sort_values(
    "MI Score",
    ascending=False
)

print("=== MUTUAL INFORMATION ===")
print("Số mẫu được khảo sát:", len(eda_sample))
print("Số đặc trưng:", X_mi.shape[1])

display(mi_result.head(20).round(5))

# Vẽ 20 đặc trưng có MI cao nhất
top20 = mi_result.head(20)

plt.figure(figsize=(12, 8))

sns.barplot(
    data=top20,
    x="MI Score",
    y="Feature",
    color="steelblue"
)

plt.title("Top 20 đặc trưng theo Mutual Information")
plt.xlabel("Mutual Information")
plt.ylabel("Đặc trưng")

plt.tight_layout()

plt.savefig(
    FIG_DIR / "04_mutual_information.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

mi_result.to_csv(
    TABLE_DIR / "04_mutual_information.csv",
    index=False,
    encoding="utf-8-sig"
)

### Nhận xét: Mutual Information

**Kết quả phân tích:**

- Các đặc trưng có điểm Mutual Information
  khác nhau, cho thấy mức độ liên hệ
  riêng lẻ với nhãn không giống nhau.
- Những đặc trưng liên quan đến payload,
  thời gian và số gói tin xuất hiện
  trong nhóm có MI cao.
- Một số đặc trưng có MI thấp, cần tiếp tục
  đánh giá mức độ hữu ích của chúng.

**Kết luận:**

Mutual Information giúp nhận diện những
đặc trưng có mối liên hệ đáng chú ý với nhãn.

Tuy nhiên, chưa thể kết luận cần loại bỏ
những đặc trưng có MI thấp.

Khi thực hiện Feature Selection thật sự,
cần tính MI trên tập Train và đánh giá
hiệu quả trên tập Validation.
